In [0]:
from pyspark.sql import functions as F

cancellation = spark.table(
    "workforce_analytics.silver.dim_cancellation_reason"
)

dim_cancellation_reason_gold = cancellation.select(
    F.col("cancellation_id"),
    F.col("cancellation_reason"),
    F.col("reason_group"),
    F.col("is_customer_driven"),
    F.col("severity")
)

(
    dim_cancellation_reason_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workforce_analytics.gold.dim_cancellation_reason_gold"
    )
)

print("dim_cancellation_reason_gold created successfully!")

In [0]:
gold_cancellation = spark.table(
    "workforce_analytics.gold.dim_cancellation_reason_gold"
)

print("Gold cancellation reason rows:", gold_cancellation.count())

display(gold_cancellation)

Next step: final Gold validation
* Before connecting Power BI, let's run one final check that all seven Gold tables exist and have the expected counts.

In [0]:
gold_tables = {
    "fact_work_gold": 9820,
    "fact_shift_gold": 9820,
    "fact_shrinkage_gold": 9820,
    "dim_people_gold": 300,
    "dim_county_gold": 10,
    "dim_shrinkage_type_gold": 5,
    "dim_cancellation_reason_gold": 10
}

for table_name, expected_count in gold_tables.items():
    df = spark.table(f"workforce_analytics.gold.{table_name}")
    actual_count = df.count()

    status = "PASS" if actual_count == expected_count else "CHECK"

    print(
        f"{table_name}: {actual_count} rows "
        f"(expected {expected_count}) — {status}"
    )